## Convolutional Neural Network

A **Convolutional Neural Network (CNN or ConvNet)** is a specialized class of deep neural network designed primarily for processing grid-like structured data—most notably **images, videos, and spatial data**.

While a standard Multi-Layer Perceptron (MLP) treats every input pixel or feature independently, CNNs are engineered to automatically and adaptively learn spatial hierarchies of features, from low-level edges and textures up to complex object shapes and faces.

---

### Why Standard MLPs Fail on Images

If you tried to feed a high-resolution color image (e.g., $224 \times 224$ pixels) into a standard MLP:

1. **Parameter Explosion:** A single hidden layer connected to every pixel would require millions of weights just for the first layer, leading to massive memory overhead and severe overfitting.
2. **Loss of Spatial Structure:** Flattening a 2D image into a 1D vector destroys the crucial spatial relationships between adjacent pixels (e.g., knowing that a pixel is right next to another pixel).

---

### Core Building Blocks of a CNN

A typical CNN architecture combines three main types of layers:

1. **Convolutional Layers (The Feature Extractors):**
* Instead of connecting every neuron, CNNs use small filters (or **kernels**, e.g., $3 \times 3$ matrices) that slide (convolve) across the image.
* By computing dot products between the filter and local patches of the image, these layers detect local patterns like edges, gradients, corners, and textures.
* **Key Advantage (Weight Sharing):** The same filter is reused across the entire image, drastically reducing the total number of parameters.


2. **Pooling Layers (Downsampling):**
* Layers like **Max Pooling** or Average Pooling reduce the spatial dimensions (height and width) of the feature maps.
* This cuts down computational complexity, controls overfitting, and provides **translation invariance** (recognizing an object whether it's in the center or corner of the frame).


3. **Fully Connected (Dense) Layers:**
* Positioned at the very end of the network (which is often an **MLP**!), these layers take the high-level abstract features extracted by the convolutional and pooling layers and output the final classification or regression prediction.

To understand how a Convolutional Neural Network (CNN) extracts patterns from an image, we need to look under the hood at its three core mechanics: the **2D Convolution Operation**, **Stride**, and **Padding**.

![Mapping an input matrix to an output feature map using a kernel. Source: Made with Google AI](https://encrypted-tbn0.gstatic.com/licensed-image?q=tbn:ANd9GcRGe4RhVwBpNkn3bP5iL0hZIhco9UEhVEdD_hotxI6zlEHNwJ8biY40fMOvhLVYTwYiAosxGkSbNcauKRM)

---

### 1. The 2D Convolution Operation

Instead of processing an entire image at once, a CNN uses a small matrix called a **filter** (or **kernel**)—typically sized $3 \times 3$ or $5 \times 5$.

This filter slides across the 2D input image. At every position, it performs an **element-wise multiplication** between its own weights and the underlying pixel values, then sums them all up into a single scalar value. This resulting matrix of values is called a **feature map**.

---

### 2. Stride ($S$)

**Stride** defines how many pixels the filter shifts horizontally or vertically at each step:

* **Stride = 1:** The filter moves 1 pixel to the right (or down), resulting in high overlap between adjacent receptive fields and preserving fine details.
* **Stride = 2 (or more):** The filter skips pixels, moving faster across the image. This shrinks the output feature map size and reduces computational cost.

---

### 3. Padding ($P$)

When a filter slides over an image, **pixels on the outer edges are touched fewer times** than pixels in the center. Furthermore, applying convolutions repeatedly causes the image dimensions to shrink rapidly layer by layer.

**Padding** solves this by adding a border of pixels (usually zeros) around the edges of the input matrix:

* **Valid Padding ($P = 0$):** No padding added. The output shrinks at every layer.
* **Same Padding ($P > 0$):** Adds enough border pixels so that the output feature map has the exact same spatial dimensions as the input.

---

### Step-by-Step Numerical Example

Let's walk through a concrete example with numbers:

* **Input Image ($W$):** $4 \times 4$ matrix
* **Kernel/Filter Size ($K$):** $3 \times 3$ matrix
* **Stride ($S$):** $1$
* **Padding ($P$):** $0$

#### The Input Matrix ($4 \times 4$)

$$\begin{bmatrix} 1 & 0 & 2 & 3 \\  4 & 6 & 1 & 0 \\  2 & 3 & 5 & 1 \\  1 & 1 & 0 & 2 \end{bmatrix}$$

#### The Kernel ($3 \times 3$)

$$\begin{bmatrix} 1 & 0 & -1 \\  1 & 1 & 0 \\  0 & -1 & 1 \end{bmatrix}$$

#### Calculation Steps:

1. **Top-Left Corner ($3 \times 3$ region):**
We place the kernel over the top-left $3 \times 3$ patch of the input:

$$\begin{bmatrix} 1 & 0 & 2 \\ 4 & 6 & 1 \\ 2 & 3 & 5 \end{bmatrix}$$



Multiply element-by-element with the kernel and sum them up:

$$(1\times1) + (0\times0) + (2\{-1\}) + (4\times1) + (6\times1) + (1\times0) + (2\times0) + (3\{-1\}) + (5\times1)$$


$$= 1 + 0 - 2 + 4 + 6 + 0 + 0 - 3 + 5 = 11$$



This gives the **first top-left value** ($11$) of our output feature map.
2. **Moving Across:**
The filter shifts right by **Stride = 1** to cover the next $3 \times 3$ patch, computes the dot product, and places the result in the top-right of the output.
3. **Moving Down:**
Once it hits the right edge, it resets to the left and shifts down by **Stride = 1**.

Because a $3 \times 3$ filter can only fit into a $4 \times 4$ grid twice horizontally and twice vertically, our output feature map will be a **$2 \times 2$ matrix**.

---

### The Universal Output Dimension Formula

To calculate the exact size of your output feature map without manual grid counting, use this standard formula:

$$\text{Output Size} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

* Where $W$ = Input width/height, $K$ = Kernel size, $P$ = Padding, $S$ = Stride.
* Plugging in our example above:

$$\left\lfloor \frac{4 - 3 + (2 \times 0)}{1} \right\rfloor + 1 = 1 + 1 = 2$$



(yielding a $2 \times 2$ output).

### Pooling Layers in CNNs: Explanation & Purpose

After a convolutional layer extracts feature maps (detailing edges, textures, and patterns), a **Pooling Layer** is typically applied to downsample those feature maps.

Just like convolutions, a pooling layer slides a window (e.g., $2 \times 2$) across the feature map with a stride, but instead of taking a weighted dot product, it performs a simple aggregation function:

1. **Max Pooling (Most Common):**
* Extracts the **maximum value** from each local pooling window.
* *Why it's used:* It acts like a noise suppressor, retaining only the strongest, most prominent features (such as a distinct edge or high-contrast corner) while discarding weaker background details.


2. **Average Pooling:**
* Calculates the **average value** of all elements inside the pooling window.
* *Why it's used:* It smooths out the feature map, preserving broader regional context rather than peak activations.



#### Why Do We Need Pooling?

* **Dimensionality Reduction:** By cutting the height and width dimensions in half (e.g., from $28 \times 28$ down to $14 \times 14$), it drastically reduces the computational cost and memory required for subsequent layers.
* **Translation Invariance:** If a feature (like a cat's ear) shifts slightly to the left or right in the image, its maximum value within the local pooling window will often remain unchanged, making the network robust to spatial shifts.
* **Expanding Receptive Field:** Downsampling allows deeper layers to look at a larger global context of the image.

---

In [1]:
import torch
import torch.nn as nn

# Define a simple CNN module
class SimpleCNNBlock(nn.Module):
    def __init__(self):
        super(SimpleCNNBlock, self).__init__()

        # 1. Convolutional Layer: 1 input channel (grayscale), 8 output filters, 3x3 kernel, padding=1
        self.conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)

        # 2. Non-linear Activation Function
        self.relu = nn.ReLU()

        # 3. Max Pooling Layer: 2x2 window with stride=2 (halves height and width)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        print(f"Original Input Shape:  {x.shape}")

        x = self.conv(x)
        print(f"After Conv2d Shape:    {x.shape} (8 channels, spatial dims unchanged due to padding=1)")

        x = self.relu(x)
        print(f"After ReLU Shape:      {x.shape}")

        x = self.pool(x)
        print(f"After MaxPool2d Shape: {x.shape} (Spatial dimensions halved: 28x28 -> 14x14)")

        return x

# Create our model
model = SimpleCNNBlock()

# Simulate a batch of 1 grayscale image (e.g., a 28x28 handwritten digit like MNIST)
dummy_image_batch = torch.randn(1, 1, 28, 28)

# Run forward pass
output_features = model(dummy_image_batch)

Original Input Shape:  torch.Size([1, 1, 28, 28])
After Conv2d Shape:    torch.Size([1, 8, 28, 28]) (8 channels, spatial dims unchanged due to padding=1)
After ReLU Shape:      torch.Size([1, 8, 28, 28])
After MaxPool2d Shape: torch.Size([1, 8, 14, 14]) (Spatial dimensions halved: 28x28 -> 14x14)


### Detailed Breakdown of the Simple CNN Block

The code block above demonstrates a single-block **Convolutional Feature Extractor**. Let's visually track how a tensor changes dimensions as it goes through each operation:

```
Input Image Batch             [Batch, Channels, Height, Width] -> [1, 1, 28, 28]
       │
       ▼
1. Conv2d (3x3, Padding=1)   -> [1, 8, 28, 28]  (Feature maps expand to 8, size stays 28x28)
       │
       ▼
2. ReLU (Non-linear)         -> [1, 8, 28, 28]  (Replaces negative activations with zero)
       │
       ▼
3. MaxPool2d (2x2, Stride=2) -> [1, 8, 14, 14]  (Reduces width & height by half)
```

#### Operational Mechanics

* **Feature Map Expansion:** By specifying `out_channels=8`, we use $8$ independent kernels (filters) in parallel. Each filter is specialized to find a unique structural feature (such as horizontal lines, vertical edges, or color transitions).
* **Preserving Resolution with Padding:** Without padding, a $3 \times 3$ kernel would shave off 2 pixels of border width and height. Setting `padding=1` surrounds our $28 \times 28$ image with a 1-pixel border of zeros, keeping the feature maps at $28 \times 28$.
* **Downsampling with Max Pooling:** The $2 \times 2$ Max Pool window looks at disjoint grids of our feature map and passes only the absolute maximum value. This extracts the most prominent features while discarding $75\%$ of the spatial data points, protecting the network from computational overload.


---

### What the Output Shows

When you run this script, you will see how the spatial dimensions transform:

* **Input:** `torch.Size([1, 1, 28, 28])` — 1 image, 1 channel, $28 \times 28$ pixels.
* **After Conv2D:** `torch.Size([1, 8, 28, 28])` — Expanded to 8 feature maps (channels) while preserving resolution via padding.
* **After MaxPool2D:** `torch.Size([1, 8, 14, 14])` — The 8 feature maps are cleanly downsampled to $14 \times 14$, cutting the spatial data size in half.

At the end of a Convolutional Neural Network's feature extraction pipeline, the network has successfully converted raw pixels into rich, abstract spatial feature maps (representing edges, shapes, and object parts).

However, **convolutional layers operate on 2D grids (channels $\times$ height $\times$ width), while classification layers (fully connected linear layers) expect a flat 1D vector of features.**

Bridging this gap requires two steps: **Flattening** and passing the result into a **Dense MLP Classification Head**.

---

### Step 1: The Output Shape After Pooling

Suppose you pass a color image of size $32 \times 32$ pixels through a CNN. After a few convolutional and max-pooling layers, your final feature map tensor might have a shape like:


$$\text{Tensor Shape} = [Batch\_Size, Channels, Height, Width] \implies [32, 32, 8, 8]$$

* **Batch Size ($32$):** Number of images processed simultaneously.
* **Channels ($32$):** Number of feature filters extracted (e.g., 32 different pattern detectors).
* **Height & Width ($8 \times 8$):** The downsampled spatial dimensions.

### Step 2: Flattening (`nn.Flatten`)

To feed this multi-dimensional tensor into a standard linear layer, we apply **Flattening**.

`nn.Flatten()` collapses all spatial and channel dimensions into a single 1D vector *for each individual image* in the batch, while keeping the batch dimension intact:


$$\text{Flattened Size} = Channels \times Height \times Width = 32 \times 8 \times 8 = 2048$$

* **New Tensor Shape:** $[32, 2048]$ (A batch of 32 images, each represented by a 2048-element feature vector).

### Step 3: The Fully Connected (Dense) Layers / MLP Head

Once flattened, this vector is passed through standard **Linear (Dense) layers**—the exact same Multi-Layer Perceptron (MLP) architecture we explored earlier:

1. **Hidden Linear Layer:** Takes the 2048 features, combines them, and maps them to a smaller dense representation (e.g., 128 neurons) with a ReLU activation to learn global combinations of features.
2. **Output Linear Layer:** Maps those dense features to the final number of target classes (e.g., 10 output nodes for a 10-class dataset like CIFAR-10) to produce raw logits.

---

### Complete PyTorch Implementation

Here is how flattening and linear classification layers are integrated at the end of a CNN in PyTorch:

In [2]:
import torch
import torch.nn as nn

class CNNWithMLPHead(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNWithMLPHead, self).__init__()

        # 1. Convolutional Feature Extractor
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 32x32 -> 16x16

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8
        )

        # 2. Fully Connected Classification Head (MLP)
        # Final spatial size: 32 channels * 8 height * 8 width = 2048 features
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # Flattens [Batch, 32, 8, 8] -> [Batch, 2048]
            nn.Linear(32 * 8 * 8, 128),      # Dense hidden layer
            nn.ReLU(),
            nn.Dropout(0.4),                  # Regularization to prevent overfitting
            nn.Linear(128, num_classes)       # Output layer (raw logits for CrossEntropyLoss)
        )

    def forward(self, x):
        # Pass through convolutions and pooling
        x = self.feature_extractor(x)

        # Pass through flatten and linear layers
        x = self.classifier(x)
        return x

# Test the model with a batch of 4 color images (3 channels, 32x32 pixels)
model = CNNWithMLPHead(num_classes=10)
dummy_images = torch.randn(4, 3, 32, 32)
output_logits = model(dummy_images)

print(f"Output Logits Shape: {output_logits.shape}")
# Expected: torch.Size([4, 10]) -> 4 images, 10 class scores each

Output Logits Shape: torch.Size([4, 10])


### Detailed Breakdown of the Complete CNN & Classifier

The model architecture above highlights how a multi-layer convolutional pipeline links to an **MLP Classifier Head** to perform categorization (such as mapping an image to 1 of 10 target classes).

Here is how the data flows structurally from raw pixel inputs to class logits:

```
   Input Tensor: [4, 3, 32, 32] (4 images, 3 channels, 32x32 pixels)
                     │
                     ▼
        [ Feature Extraction Pipeline ]
   - Conv2d (16 filters) & MaxPool  --> [4, 16, 16, 16]
   - Conv2d (32 filters) & MaxPool  --> [4, 32, 8, 8]
                     │
                     ▼
            [ nn.Flatten() ]
   - Collapse channels & spatial dimensions: 32 x 8 x 8
   - Output Shape                   --> [4, 2048]
                     │
                     ▼
         [ MLP Classification Head ]
   - Linear Layer (2048 -> 128)     --> [4, 128]
   - ReLU & Dropout (0.4)           --> [4, 128]
   - Linear Layer (128 -> 10)       --> [4, 10]
                     │
                     ▼
   Output Logits: [4, 10] (10 class probabilities/scores per image)
```

#### Key Architectural Choices Explained

1. **Progressive Channel Expansion:** As we go deeper into the network, the spatial size decreases ($32 \to 16 \to 8$), but the number of channels increases ($3 \to 16 \to 32$). This helps the model compress the image's raw spatial coordinates into abstract high-level semantical representations.
2. **The Dropout Layer (`nn.Dropout(0.4)`):** Modern deep neural networks have millions of parameters. Dropout randomly deactivates $40\%$ of the neurons in the classifier head during training, forcing the network to learn robust, generalized representations instead of memorizing the training dataset.